# AMD ROCm Quick Start 🚀

**目标: 5 分钟内验证 ROCm 环境，跑通 PyTorch GPU 张量运算**

适用: AMD GPU (RX 6000+ / MI 系列) + ROCm 6.x + PyTorch 2.x

## 基础环境配置检查
### 0.1 Install ROCm + PyTorch

```bash
# Ubuntu 22.04 / 24.04: Install ROCm 7.0
wget https://repo.radeon.com/amdgpu-install/latest/ubuntu/jammy/amdgpu-install_6.4.60400-1_all.deb
sudo apt install ./amdgpu-install_6.4.60400-1_all.deb
sudo amdgpu-install --usecase=rocmdev

# Install PyTorch ROCm version
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm7.0

# Verify
rocm-smi
python -c "import torch; print(torch.cuda.is_available())"  # True
```

### 0.2 Python Environment (requirements.txt)

```
torch>=2.4.0
accelerate>=0.34.0
bitsandbytes>=0.43.0
vllm>=0.6.0
matplotlib>=3.8.0
jupyter>=1.0.0
nbformat>=5.10.0
```

For ROCm PyTorch, install from:
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm7.0
```

In [1]:
# Cell 1: Check System & ROCm Driver
# 检查 ROCm 驱动和 GPU 信息

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip() or result.stderr.strip()

print("=" * 50)
print("System Info")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")

# Check rocm-smi (ROCm System Management Interface)
try:
    smi_out = run("rocm-smi --showproductname 2>/dev/null || echo 'rocm-smi not found'")
    print(f"\nGPU Info:\n{smi_out}")
except:
    print("rocm-smi: NOT FOUND - please install ROCm driver")

# Check ROCm version
try:
    rocm_ver = run("cat /opt/rocm/.info/version 2>/dev/null || echo 'N/A'")
    print(f"ROCm Version: {rocm_ver}")
except:
    print("ROCm Version: N/A")

System Info
Python: 3.12.13

GPU Info:
============================ ROCm System Management Interface ============================
====================================== Product Info ======================================
GPU[0]		: get_name, Error when calling libdrm
GPU[0]		: Card Series: 		N/A
GPU[0]		: Card Model: 		0x74b6
GPU[0]		: Card Vendor: 		Advanced Micro Devices, Inc. [AMD/ATI]
GPU[0]		: Card SKU: 		M3080202
GPU[0]		: Subsystem ID: 	0x74a2
GPU[0]		: Device Rev: 		0x00
GPU[0]		: Node ID: 		1
GPU[0]		: GUID: 		6100
GPU[0]		: GFX Version: 		gfx942
================================== End of ROCm SMI Log ===================================
ROCm Version: 7.2.1


In [2]:
# Cell 2: Check PyTorch with ROCm Support
# 验证 PyTorch 是否编译了 ROCm 后端

import torch

print("=" * 50)
print("PyTorch & ROCm Status")
print("=" * 50)

print(f"PyTorch Version: {torch.__version__}")
print(f"ROCm Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        # print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  GPU {i} - 架构型号: {torch.cuda.get_device_properties(i).gcnArchName}")
        print(f"  GPU {i} - 显存大小: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
        print(f"  GPU {i} - 计算单元: {torch.cuda.get_device_properties(i).multi_processor_count}")
    print(f"ROCm Version (PyTorch): {torch.version.hip}")
    print(f"CUDA Arch List: {torch.cuda.get_arch_list()}")
else:
    print("\n[DEMO MODE] No ROCm GPU detected. Running on CPU for demo.")

PyTorch & ROCm Status
PyTorch Version: 2.10.0+git8514f05
ROCm Available: True
GPU Count: 1
  GPU 0 - 架构型号: gfx942:sramecc+:xnack-
  GPU 0 - 显存大小: 191.69 GB
  GPU 0 - 计算单元: 80
ROCm Version (PyTorch): 7.2.53211
CUDA Arch List: ['gfx90a', 'gfx942', 'gfx950', 'gfx1100', 'gfx1101', 'gfx1200', 'gfx1201', 'gfx1150', 'gfx1151']


In [3]:
# Cell 3: Tensor Operations on GPU
# 在 GPU 上做基础张量运算，对比 CPU 性能

import torch
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---- 3.1 矩阵乘法 ----
N = 4096
a = torch.randn(N, N, dtype=torch.float32)
b = torch.randn(N, N, dtype=torch.float32)

# CPU timing
t0 = time.time()
c_cpu = a @ b
t_cpu = time.time() - t0
print(f"CPU matmul {N}x{N}: {t_cpu:.3f}s")

# GPU timing
a_gpu, b_gpu = a.to(device), b.to(device)
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
c_gpu = a_gpu @ b_gpu
if device.type == "cuda":
    torch.cuda.synchronize()
t_gpu = time.time() - t0
print(f"GPU matmul {N}x{N}: {t_gpu:.3f}s")

if device.type == "cuda" and t_gpu > 0:
    print(f"Speedup: {t_cpu / t_gpu:.1f}x")

# ---- 3.2 内存传输开销 ----
size = 1024 * 1024 * 100  # 100M elements (~400MB)
data = torch.randn(size)
t0 = time.time()
data_gpu = data.to(device)
if device.type == "cuda":
    torch.cuda.synchronize()
t_upload = time.time() - t0
print(f"\nUpload 400MB to GPU: {t_upload*1000:.1f}ms")

Using device: cuda
CPU matmul 4096x4096: 10.331s
GPU matmul 4096x4096: 0.775s
Speedup: 13.3x

Upload 400MB to GPU: 12.7ms


In [4]:
# Cell 4: Simple Neural Network on GPU
# 搭建一个简单 MLP，在 GPU 上训练几步验证端到端流程

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define a simple 3-layer MLP
class SimpleMLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)

model = SimpleMLP().to(device)
print(f"Model on: {next(model.parameters()).device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Simulated training loop (3 steps)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("\nTraining 3 steps (simulated data)...")
for step in range(3):
    x = torch.randn(64, 784).to(device)  # batch=64
    # y = torch.randint(0, 10, (64,)).to(device) # 演示不太合理（随机难学习，loss不下降）
    y = torch.clamp((x.abs().mean(dim=1) * 10).long(), 0, 9).to(device)
    
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    print(f"  Step {step+1}: loss = {loss.item():.4f}")

print("\nAll good! ROCm + PyTorch is ready.")

Model on: cuda:0
Parameters: 269,322

Training 3 steps (simulated data)...
  Step 1: loss = 2.2882
  Step 2: loss = 2.1302
  Step 3: loss = 1.9738

All good! ROCm + PyTorch is ready.


## Summary

✅ 验证了 ROCm 驱动和 GPU 信息  
✅ PyTorch 正确识别 ROCm 后端  
✅ GPU 张量运算正常（矩阵乘法加速）  
✅ 简单神经网络可在 ROCm 上训练  

**下一步**: 试试 `AMD_ROCm_LLM_Inference.ipynb` 跑 LLM 推理！